# Multimodal Learning for Hotel Price Prediction — Supervisor-Corrected Version

This notebook is reconstructed from the supplied HTML export and includes the requested corrections:

- train-only fitting of categorical encoding, imputation and scaling;
- XGBoost history-only using the same five price lags as LSTM;
- validation-selected XGBoost–LSTM late fusion;
- general and FRED ablation figures;
- identical y-axis ranges for independent prediction/error plots;
- chronological ordering for visualisation without incorrectly calling the plot a continuous time series.

Run the notebook from the first cell after placing the hotel and FRED CSV files in the working directory.


In [ ]:
import os
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import glob
import warnings
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

features = pd.read_csv("hotels-europe_features.csv")
price = pd.read_csv("hotels-europe_price.csv")

df_full = pd.merge(features, price, on="hotel_id")
df_full = df_full.sort_values(["hotel_id", "year", "month"]).copy()

sequence_length = 5
prediction_horizon = 1

print(f"Prediction setting: use past {sequence_length} month(s) to predict {prediction_horizon} month(s) ahead.")
print("Raw merged data shape:", df_full.shape)


In [ ]:
from pathlib import Path
import warnings

FRED_LOCAL_DIR = Path(".") 
MIN_RECOMMENDED_FRED_INDEX_COUNT = 20

SAVE_OUTPUT_CSV_FILES = False

print("Current working directory:", Path.cwd())
print("FRED local directory:", FRED_LOCAL_DIR.resolve())

fred_config = {
     # Price level / inflation
    "inflation_cpi": {
        "category": "price_level_inflation",
        "series_ids": ["CPIAUCSL"],
        "file_patterns": ["CPIAUCSL.csv", "CPIAUCSL*.csv"],
        "raw_col": "inflation_raw",
        "index_col": "inflation_index",
        "description": "Consumer Price Index for All Urban Consumers",
        "required": True
    },
    "core_cpi": {
        "category": "price_level_inflation",
        "series_ids": ["CPILFESL"],
        "file_patterns": ["CPILFESL.csv", "CPILFESL*.csv"],
        "raw_col": "core_cpi_raw",
        "index_col": "core_cpi_index",
        "description": "Core CPI excluding food and energy",
        "required": False
    },
    "pce_price_index": {
        "category": "price_level_inflation",
        "series_ids": ["PCEPI"],
        "file_patterns": ["PCEPI.csv", "PCEPI*.csv"],
        "raw_col": "pce_price_index_raw",
        "index_col": "pce_price_index_index",
        "description": "Personal Consumption Expenditures Price Index",
        "required": False
    },
    "core_pce_price_index": {
        "category": "price_level_inflation",
        "series_ids": ["PCEPILFE"],
        "file_patterns": ["PCEPILFE.csv", "PCEPILFE*.csv"],
        "raw_col": "core_pce_price_index_raw",
        "index_col": "core_pce_price_index_index",
        "description": "Core PCE Price Index",
        "required": False
    },
    "producer_price_index": {
        "category": "price_level_inflation",
        "series_ids": ["PPIACO"],
        "file_patterns": ["PPIACO.csv", "PPIACO*.csv"],
        "raw_col": "producer_price_index_raw",
        "index_col": "producer_price_index_index",
        "description": "Producer Price Index: All Commodities",
        "required": False
    },

    # Energy / operating cost
    "energy_cpi": {
        "category": "energy_operating_cost",
        "series_ids": ["CPIENGSL"],
        "file_patterns": ["CPIENGSL.csv", "CPIENGSL*.csv"],
        "raw_col": "energy_cpi_raw",
        "index_col": "energy_cpi_index",
        "description": "CPI Energy",
        "required": False
    },
    "fuel_price": {
        "category": "energy_operating_cost",
        "series_ids": ["MCOILWTICO", "DCOILWTICO"],
        "file_patterns": ["MCOILWTICO.csv", "MCOILWTICO*.csv", "DCOILWTICO.csv", "DCOILWTICO*.csv"],
        "raw_col": "fuel_price_raw",
        "index_col": "fuel_price_index",
        "description": "WTI crude oil price, fuel-price proxy",
        "required": True
    },
    "gasoline_price": {
        "category": "energy_operating_cost",
        "series_ids": ["GASREGW"],
        "file_patterns": ["GASREGW.csv", "GASREGW*.csv"],
        "raw_col": "gasoline_price_raw",
        "index_col": "gasoline_price_index",
        "description": "US regular gasoline price",
        "required": False
    },
    "natural_gas_price": {
        "category": "energy_operating_cost",
        "series_ids": ["DHHNGSP"],
        "file_patterns": ["DHHNGSP.csv", "DHHNGSP*.csv"],
        "raw_col": "natural_gas_price_raw",
        "index_col": "natural_gas_price_index",
        "description": "Henry Hub natural gas spot price",
        "required": False
    },
    "electricity": {
        "category": "energy_operating_cost",
        "series_ids": ["CUSR0000SEHF01"],
        "file_patterns": ["CUSR0000SEHF01.csv", "CUSR0000SEHF01*.csv"],
        "raw_col": "electricity_raw",
        "index_col": "electricity_index",
        "description": "CPI for electricity",
        "required": True
    },

    # Consumer demand / spending
    "personal_consumption": {
        "category": "consumer_demand",
        "series_ids": ["PCE"],
        "file_patterns": ["PCE.csv", "PCE*.csv"],
        "raw_col": "personal_consumption_raw",
        "index_col": "personal_consumption_index",
        "description": "Personal Consumption Expenditures",
        "required": False
    },
    "services_consumption": {
        "category": "consumer_demand",
        "series_ids": ["PCES"],
        "file_patterns": ["PCES.csv", "PCES*.csv"],
        "raw_col": "services_consumption_raw",
        "index_col": "services_consumption_index",
        "description": "Personal Consumption Expenditures: Services",
        "required": False
    },
    "real_disposable_income": {
        "category": "consumer_demand",
        "series_ids": ["DSPIC96"],
        "file_patterns": ["DSPIC96.csv", "DSPIC96*.csv"],
        "raw_col": "real_disposable_income_raw",
        "index_col": "real_disposable_income_index",
        "description": "Real Disposable Personal Income",
        "required": False
    },
    "personal_saving_rate": {
        "category": "consumer_demand",
        "series_ids": ["PSAVERT"],
        "file_patterns": ["PSAVERT.csv", "PSAVERT*.csv"],
        "raw_col": "personal_saving_rate_raw",
        "index_col": "personal_saving_rate_index",
        "description": "Personal Saving Rate",
        "required": False
    },
    "retail_sales": {
        "category": "consumer_demand",
        "series_ids": ["RSAFS"],
        "file_patterns": ["RSAFS.csv", "RSAFS*.csv"],
        "raw_col": "retail_sales_raw",
        "index_col": "retail_sales_index",
        "description": "Advance Retail Sales: Retail and Food Services",
        "required": False
    },
    "retail_sales_ex_food": {
        "category": "consumer_demand",
        "series_ids": ["RSXFS"],
        "file_patterns": ["RSXFS.csv", "RSXFS*.csv"],
        "raw_col": "retail_sales_ex_food_raw",
        "index_col": "retail_sales_ex_food_index",
        "description": "Retail Sales excluding Food Services",
        "required": False
    },
    "consumer_sentiment": {
        "category": "consumer_demand",
        "series_ids": ["UMCSENT"],
        "file_patterns": ["UMCSENT.csv", "UMCSENT*.csv"],
        "raw_col": "consumer_sentiment_raw",
        "index_col": "consumer_sentiment_index",
        "description": "University of Michigan Consumer Sentiment",
        "required": False
    },

    # Labour market / wage pressure
    "unemployment_rate": {
        "category": "labour_market",
        "series_ids": ["UNRATE"],
        "file_patterns": ["UNRATE.csv", "UNRATE*.csv"],
        "raw_col": "unemployment_rate_raw",
        "index_col": "unemployment_rate_index",
        "description": "Unemployment Rate",
        "required": False
    },
    "nonfarm_payrolls": {
        "category": "labour_market",
        "series_ids": ["PAYEMS"],
        "file_patterns": ["PAYEMS.csv", "PAYEMS*.csv"],
        "raw_col": "nonfarm_payrolls_raw",
        "index_col": "nonfarm_payrolls_index",
        "description": "Total Nonfarm Payrolls",
        "required": False
    },
    "avg_hourly_earnings": {
        "category": "labour_market",
        "series_ids": ["CES0500000003"],
        "file_patterns": ["CES0500000003.csv", "CES0500000003*.csv"],
        "raw_col": "avg_hourly_earnings_raw",
        "index_col": "avg_hourly_earnings_index",
        "description": "Average Hourly Earnings, Total Private",
        "required": False
    },
    "initial_claims": {
        "category": "labour_market",
        "series_ids": ["ICSA"],
        "file_patterns": ["ICSA.csv", "ICSA*.csv"],
        "raw_col": "initial_claims_raw",
        "index_col": "initial_claims_index",
        "description": "Initial Unemployment Claims",
        "required": False
    },
    "job_openings": {
        "category": "labour_market",
        "series_ids": ["JTSJOL"],
        "file_patterns": ["JTSJOL.csv", "JTSJOL*.csv"],
        "raw_col": "job_openings_raw",
        "index_col": "job_openings_index",
        "description": "Job Openings: Total Nonfarm",
        "required": False
    },

    # Financial conditions
    "fed_funds_rate": {
        "category": "financial_conditions",
        "series_ids": ["FEDFUNDS"],
        "file_patterns": ["FEDFUNDS.csv", "FEDFUNDS*.csv"],
        "raw_col": "fed_funds_rate_raw",
        "index_col": "fed_funds_rate_index",
        "description": "Effective Federal Funds Rate",
        "required": False
    },
    "ten_year_treasury": {
        "category": "financial_conditions",
        "series_ids": ["DGS10"],
        "file_patterns": ["DGS10.csv", "DGS10*.csv"],
        "raw_col": "ten_year_treasury_raw",
        "index_col": "ten_year_treasury_index",
        "description": "10-Year Treasury Constant Maturity Rate",
        "required": False
    },
    "two_year_treasury": {
        "category": "financial_conditions",
        "series_ids": ["DGS2"],
        "file_patterns": ["DGS2.csv", "DGS2*.csv"],
        "raw_col": "two_year_treasury_raw",
        "index_col": "two_year_treasury_index",
        "description": "2-Year Treasury Constant Maturity Rate",
        "required": False
    },
    "mortgage_rate": {
        "category": "financial_conditions",
        "series_ids": ["MORTGAGE30US"],
        "file_patterns": ["MORTGAGE30US.csv", "MORTGAGE30US*.csv"],
        "raw_col": "mortgage_rate_raw",
        "index_col": "mortgage_rate_index",
        "description": "30-Year Fixed Rate Mortgage Average",
        "required": False
    },
    "financial_volatility": {
        "category": "financial_conditions",
        "series_ids": ["VIXCLS"],
        "file_patterns": ["VIXCLS.csv", "VIXCLS*.csv"],
        "raw_col": "financial_volatility_raw",
        "index_col": "financial_volatility_index",
        "description": "CBOE Volatility Index",
        "required": False
    },
    "trade_weighted_dollar": {
        "category": "financial_conditions",
        "series_ids": ["DTWEXBGS"],
        "file_patterns": ["DTWEXBGS.csv", "DTWEXBGS*.csv"],
        "raw_col": "trade_weighted_dollar_raw",
        "index_col": "trade_weighted_dollar_index",
        "description": "Trade Weighted U.S. Dollar Index",
        "required": False
    },

    # Real activity
    "industrial_production": {
        "category": "real_activity",
        "series_ids": ["INDPRO"],
        "file_patterns": ["INDPRO.csv", "INDPRO*.csv"],
        "raw_col": "industrial_production_raw",
        "index_col": "industrial_production_index",
        "description": "Industrial Production Index",
        "required": False
    },
    "housing_starts": {
        "category": "real_activity",
        "series_ids": ["HOUST"],
        "file_patterns": ["HOUST.csv", "HOUST*.csv"],
        "raw_col": "housing_starts_raw",
        "index_col": "housing_starts_index",
        "description": "Housing Starts",
        "required": False
    },
}

fred_download_checklist = pd.DataFrame([
    {
        "indicator_key": key,
        "category": cfg["category"],
        "series_ids": " or ".join(cfg["series_ids"]),
        "required": cfg["required"],
        "description": cfg["description"],
        "recommended_filename": cfg["file_patterns"][0],
        "download_url": f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={cfg['series_ids'][0]}"
    }
    for key, cfg in fred_config.items()
])
if SAVE_OUTPUT_CSV_FILES:
    fred_download_checklist.to_csv("fred_20_30_index_download_checklist.csv", index=False)
    print("FRED download checklist saved to fred_20_30_index_download_checklist.csv")
else:
    print("FRED download checklist is displayed below; no CSV file is saved.")
display(fred_download_checklist.head(10))


def find_local_fred_file(cfg, local_dir=FRED_LOCAL_DIR):
    """
    Find the first local CSV file matching the expected FRED file pattern.
    If filenames were not renamed, it also scans CSV headers for the FRED series ID.
    """
    local_dir = Path(local_dir)
    matches = []

    for pattern in cfg["file_patterns"]:
        matches.extend(sorted(local_dir.glob(pattern)))

    # Remove duplicates while preserving order.
    unique_matches = []
    seen = set()
    for path in matches:
        if path not in seen:
            unique_matches.append(path)
            seen.add(path)

    if unique_matches:
        return unique_matches[0]

    for csv_path in sorted(local_dir.glob("*.csv")):
        try:
            header = pd.read_csv(csv_path, nrows=0).columns.astype(str).tolist()
        except Exception:
            continue
        if any(series_id in header for series_id in cfg["series_ids"]):
            return csv_path

    return None


def standardize_local_fred_dataframe(temp, series_ids, raw_col, source_path):
    """
    Standardize a FRED CSV into two columns:
        date, raw_col

    Works for common FRED formats:
        DATE, CPIAUCSL
        observation_date, CPIAUCSL
        date, value
    """
    temp = temp.copy()
    temp.columns = [str(c).strip() for c in temp.columns]

    date_candidates = [c for c in temp.columns if c.lower() in ["date", "observation_date"]]
    if date_candidates:
        date_col = date_candidates[0]
    else:
        date_col = temp.columns[0]

    value_col = None
    for sid in series_ids:
        if sid in temp.columns:
            value_col = sid
            break

    if value_col is None:
        non_date_cols = [c for c in temp.columns if c != date_col]
        if not non_date_cols:
            raise ValueError(f"No value column found in {source_path}")
        value_col = non_date_cols[0]

    temp = temp[[date_col, value_col]].rename(columns={date_col: "date", value_col: raw_col})
    temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
    temp[raw_col] = pd.to_numeric(temp[raw_col], errors="coerce")
    temp = temp.dropna(subset=["date"])
    temp = temp.sort_values("date").drop_duplicates(subset=["date"], keep="last")
    return temp


def load_fred_series_from_local_csv(cfg, start_date, end_date):
    """Load one FRED series from local CSV. Returns None when an optional file is missing."""
    source_path = find_local_fred_file(cfg)

    if source_path is None:
        return None, None

    print(f"Loading local FRED file for {cfg['description']}: {source_path}")
    temp = pd.read_csv(source_path)
    temp = standardize_local_fred_dataframe(
        temp=temp,
        series_ids=cfg["series_ids"],
        raw_col=cfg["raw_col"],
        source_path=source_path
    )

    # Restrict to study period, with buffer for YoY calculation.
    temp = temp[
        (temp["date"] >= pd.Timestamp(start_date)) &
        (temp["date"] <= pd.Timestamp(end_date))
    ]

    if temp.empty:
        warnings.warn(
            f"{source_path} has no observations between "
            f"{pd.Timestamp(start_date).date()} and {pd.Timestamp(end_date).date()}. "
            "This optional FRED series will be skipped."
        )
        return None, source_path.name

    # Convert daily, weekly, monthly, or quarterly data to one monthly value.
    # Daily/weekly series become monthly averages. Quarterly series are forward-filled after resampling.
    monthly = temp.set_index("date")[[cfg["raw_col"]]].resample("MS").mean()
    monthly[cfg["raw_col"]] = monthly[cfg["raw_col"]].ffill().bfill()
    monthly = monthly.reset_index()

    if monthly[cfg["raw_col"]].dropna().empty:
        warnings.warn(f"{source_path} has no valid numeric values after cleaning. It will be skipped.")
        return None, source_path.name

    print(f"Loaded {source_path.name}. Monthly rows: {len(monthly)}")
    return monthly, source_path.name


def build_fred_monthly_features(start_date, end_date):
    """
    Build monthly FRED index features and percentage-change features from local CSV files.

    For each loaded FRED series:
        raw value      -> used only to construct index
        index value    -> base 100 at first valid observation in the study window
        MoM change     -> pct_change(1)
        YoY change     -> pct_change(12)
    """
    frames = []
    load_status = []
    missing_required = []

    for name, cfg in fred_config.items():
        raw_col = cfg["raw_col"]
        index_col = cfg["index_col"]

        temp, used_file = load_fred_series_from_local_csv(
            cfg=cfg,
            start_date=start_date,
            end_date=end_date
        )

        if temp is None:
            load_status.append({
                "indicator_key": name,
                "category": cfg["category"],
                "loaded": False,
                "used_file": used_file,
                "required": cfg["required"],
                "series_ids": " or ".join(cfg["series_ids"]),
                "description": cfg["description"],
                "download_url": f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={cfg['series_ids'][0]}"
            })
            if cfg["required"]:
                missing_required.append(name)
            continue

        first_valid_series = temp[raw_col].dropna()
        first_valid = first_valid_series.iloc[0]

        # Convert raw series to an index, base = first available value in the study window.
        temp[index_col] = temp[raw_col] / first_valid * 100

        # Create month-on-month and year-on-year percentage changes.
        temp[f"{index_col}_mom"] = temp[index_col].pct_change()
        temp[f"{index_col}_yoy"] = temp[index_col].pct_change(12)

        frames.append(temp[["date", index_col, f"{index_col}_mom", f"{index_col}_yoy"]])

        load_status.append({
            "indicator_key": name,
            "category": cfg["category"],
            "loaded": True,
            "used_file": used_file,
            "required": cfg["required"],
            "series_ids": " or ".join(cfg["series_ids"]),
            "description": cfg["description"],
            "index_col": index_col,
            "mom_col": f"{index_col}_mom",
            "yoy_col": f"{index_col}_yoy",
            "download_url": f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={cfg['series_ids'][0]}"
        })

    fred_load_status_df = pd.DataFrame(load_status)
    if SAVE_OUTPUT_CSV_FILES:
        fred_load_status_df.to_csv("fred_load_status.csv", index=False)

    missing_optional_df = fred_load_status_df[fred_load_status_df["loaded"] == False].copy()
    if SAVE_OUTPUT_CSV_FILES:
        missing_optional_df.to_csv("fred_missing_optional_download_list.csv", index=False)

    if missing_required:
        required_message = fred_load_status_df[
            (fred_load_status_df["indicator_key"].isin(missing_required))
        ][["indicator_key", "series_ids", "download_url"]]
        raise FileNotFoundError(
            "Required FRED CSV files are missing. Please download these files into the notebook folder:\n"
            + required_message.to_string(index=False)
        )

    if not frames:
        raise FileNotFoundError(
            "No FRED CSV files were found. Please place the FRED CSV files in the same folder as this notebook."
        )

    fred_df = frames[0]
    for frame in frames[1:]:
        fred_df = fred_df.merge(frame, on="date", how="outer")

    fred_df = fred_df.sort_values("date").copy()
    fred_df["year"] = fred_df["date"].dt.year
    fred_df["month"] = fred_df["date"].dt.month

    return fred_df.drop(columns=["date"]), fred_load_status_df


# The buffer supports YoY calculations before the first hotel observation month.
start_date = pd.Timestamp(int(df_full["year"].min()), int(df_full["month"].min()), 1) - pd.DateOffset(months=18)
end_date = pd.Timestamp(int(df_full["year"].max()), int(df_full["month"].max()), 1) + pd.DateOffset(months=1)

print("Study FRED date range:", start_date.date(), "to", end_date.date())

fred_monthly, fred_load_status_df = build_fred_monthly_features(start_date, end_date)

loaded_fred_status = fred_load_status_df[fred_load_status_df["loaded"] == True].copy()
loaded_indicator_keys = loaded_fred_status["indicator_key"].tolist()

print(f"Loaded FRED indicators: {len(loaded_indicator_keys)}")
print("Loaded indicator keys:", loaded_indicator_keys)

if len(loaded_indicator_keys) < MIN_RECOMMENDED_FRED_INDEX_COUNT:
    warnings.warn(
        f"Only {len(loaded_indicator_keys)} FRED indicator(s) were loaded. "
        f"The supervisor's '20-30 index' note is better addressed by downloading at least "
        f"{MIN_RECOMMENDED_FRED_INDEX_COUNT} CSV files from fred_20_30_index_download_checklist.csv. "
        "The notebook will still run with the available indicators."
    )

# Merge FRED monthly features by year and month.
df_full = df_full.merge(fred_monthly, on=["year", "month"], how="left")

# Build feature groups dynamically based on the FRED files actually found.
fred_index_cols = []
fred_change_cols = []
fred_indicator_feature_groups = {}
fred_category_index_groups = {}
fred_category_all_feature_groups = {}

for _, row in loaded_fred_status.iterrows():
    key = row["indicator_key"]
    cfg = fred_config[key]
    index_col = cfg["index_col"]
    mom_col = f"{index_col}_mom"
    yoy_col = f"{index_col}_yoy"

    candidate_cols = [index_col, mom_col, yoy_col]
    existing_cols = [c for c in candidate_cols if c in df_full.columns]

    if index_col in df_full.columns:
        fred_index_cols.append(index_col)
    fred_change_cols.extend([c for c in [mom_col, yoy_col] if c in df_full.columns])
    fred_indicator_feature_groups[key] = existing_cols

    category = cfg["category"]
    fred_category_index_groups.setdefault(category, [])
    fred_category_all_feature_groups.setdefault(category, [])

    if index_col in df_full.columns:
        fred_category_index_groups[category].append(index_col)
    fred_category_all_feature_groups[category].extend(existing_cols)


core_indicator_keys = ["inflation_cpi", "fuel_price", "electricity"]
core_fred_change_cols = []
for key in core_indicator_keys:
    for c in fred_indicator_feature_groups.get(key, [])[1:]:
        if c in df_full.columns:
            core_fred_change_cols.append(c)

# This is the feature set used by the main model.
exogenous_cols = fred_index_cols + core_fred_change_cols
exogenous_cols = [c for c in exogenous_cols if c in df_full.columns]

# This is the larger feature pool used for expanded ablation experiments.
fred_all_feature_cols = fred_index_cols + fred_change_cols
fred_all_feature_cols = [c for c in fred_all_feature_cols if c in df_full.columns]

# Compatibility aliases for the original 3 variables.
# If the original column names are expected elsewhere, these names remain available:
#   inflation_index, fuel_price_index, and electricity_index keep the original core names.

# Clean possible missing values in FRED features created at the beginning of the period.
df_full[fred_all_feature_cols] = df_full[fred_all_feature_cols].replace([np.inf, -np.inf], np.nan)
df_full[fred_all_feature_cols] = df_full[fred_all_feature_cols].ffill().bfill()

print("FRED monthly features shape:", fred_monthly.shape)
print("Main exogenous columns used by model:", exogenous_cols)
print("Full FRED feature pool for ablation:", len(fred_all_feature_cols), "columns")
print("FRED category groups:")
for category, cols in fred_category_index_groups.items():
    print(f"  {category}: {len(cols)} index column(s)")

display(loaded_fred_status[["indicator_key", "category", "series_ids", "used_file", "description"]])
print(df_full[["year", "month"] + exogenous_cols].drop_duplicates().head(15))


In [ ]:
base_tabular_cols = [
    "distance", "stars", "rating", "rating_reviewcount",
    "ratingta", "ratingta_count", "distance_alter",
    "city", "accommodation_type", "offer", "offer_cat",
    "year", "month", "weekend", "holiday", "nnights", "scarce_room"
]

base_tabular_cols = [c for c in base_tabular_cols if c in df_full.columns]

# The final tabular model uses original hotel/search/date variables + expanded FRED index variables.
tabular_cols = base_tabular_cols + exogenous_cols

rows = []

for hotel_id, group in df_full.groupby("hotel_id"):
    group = group.sort_values(["year", "month"]).reset_index(drop=True)

    if len(group) <= sequence_length + prediction_horizon - 1:
        continue

    for i in range(len(group) - sequence_length - prediction_horizon + 1):
        history_window = group.loc[i:i + sequence_length - 1]
        forecast_origin_row = group.loc[i + sequence_length - 1]
        target_row = group.loc[i + sequence_length + prediction_horizon - 1]

        hist_prices = history_window["price"].values

        row = {
            "hotel_id": hotel_id,
            "target_price": target_row["price"],
            "target_year": target_row["year"],
            "target_month": target_row["month"],
            "origin_year": forecast_origin_row["year"],
            "origin_month": forecast_origin_row["month"],
            "prediction_horizon": prediction_horizon
        }

        for j in range(sequence_length):
            row[f"hist_price_{j + 1}"] = hist_prices[j]

        # Hotel/search/date variables are taken from the target row because these are the row-level
        # covariates used in the original model. Month/year are known calendar features.
        for col in base_tabular_cols:
            row[col] = target_row[col]

        # FRED index variables are taken from the forecast origin month to avoid future information leakage.
        for col in exogenous_cols:
            row[col] = forecast_origin_row[col]

        rows.append(row)

aligned_df = pd.DataFrame(rows)

print("Aligned supervised data shape:", aligned_df.shape)
print("Base tabular columns:", base_tabular_cols)
print("Expanded FRED columns used by the main model:", exogenous_cols)
print(aligned_df.head())


## Prepare model inputs

This section prepares three aligned inputs:

- **Historical lag matrix:** the previous five monthly prices, used by XGBoost.
- **Sequential input:** the same five prices reshaped as an ordered sequence for LSTM.
- **Raw tabular input:** hotel/search/date features plus FRED variables. Categorical encoding, imputation and scaling are fitted on training data only after the outer split.


In [ ]:
hist_price_cols = [f"hist_price_{i + 1}" for i in range(sequence_length)]

# The same five historical prices are represented in two ways:
#   1. a flat lag matrix for XGBoost;
#   2. an ordered 3-D sequence for LSTM.
X_hist = aligned_df[hist_price_cols].to_numpy(dtype=float)
X_seq = X_hist.reshape(X_hist.shape[0], sequence_length, 1)

y_seq = aligned_df["target_price"].to_numpy(dtype=float)
y_tab = y_seq.copy()

# Keep the tabular data RAW here. One-hot encoding, imputation and scaling are
# fitted after the train/test split, using outer-training rows only.
X_tab_raw = aligned_df[tabular_cols].copy()
X_tab_raw = X_tab_raw.replace([np.inf, -np.inf], np.nan)

categorical_cols = [
    col for col in ["city", "accommodation_type", "offer", "offer_cat"]
    if col in X_tab_raw.columns and (
        X_tab_raw[col].dtype == "object"
        or str(X_tab_raw[col].dtype) == "category"
    )
]

print("X_hist shape:", X_hist.shape)
print("X_seq shape:", X_seq.shape)
print("y shape:", y_seq.shape)
print("Raw X_tab shape before train-only preprocessing:", X_tab_raw.shape)
print("Categorical columns:", categorical_cols)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from keras.models import Sequential, Model
from keras.layers import Input, Dense, LSTM, Concatenate, Add, Dropout
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    nonzero_mask = y_true != 0
    mape = np.mean(
        np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])
    ) * 100
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE": mape,
        "R2": r2_score(y_true, y_pred),
    }


def make_dense_onehot_encoder():
    """Return a dense OneHotEncoder compatible with old and new sklearn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_tabular_preprocessor(X_raw):
    """
    Build a leakage-safe tabular preprocessor.

    The returned object must be fitted on training rows only. Numeric variables
    are median-imputed and standardised; categorical variables are imputed and
    one-hot encoded with unknown-category handling.
    """
    cat_cols = [
        c for c in X_raw.columns
        if X_raw[c].dtype == "object" or str(X_raw[c].dtype) == "category"
    ]
    num_cols = [c for c in X_raw.columns if c not in cat_cols]

    transformers = []
    if num_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ))
    if cat_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_dense_onehot_encoder()),
            ]),
            cat_cols,
        ))

    if not transformers:
        raise ValueError("No tabular columns are available for preprocessing.")

    return ColumnTransformer(transformers=transformers, remainder="drop")


# -------------------------------------------------------------------------
# Leakage-safe outer holdout split
# -------------------------------------------------------------------------
# The test set is created once and is not used for feature selection,
# hyperparameter selection, fusion-weight selection, or preprocessing fitting.
indices = np.arange(len(aligned_df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

# Fit categorical vocabulary, imputers and numeric scalers on outer-training
# rows only. Test rows are transformed with the fitted preprocessor.
main_tabular_preprocessor = build_tabular_preprocessor(X_tab_raw)
X_train_tab = main_tabular_preprocessor.fit_transform(X_tab_raw.iloc[train_idx])
X_test_tab = main_tabular_preprocessor.transform(X_tab_raw.iloc[test_idx])
X_train_tab = np.asarray(X_train_tab, dtype=np.float32)
X_test_tab = np.asarray(X_test_tab, dtype=np.float32)

try:
    main_tabular_feature_names = list(main_tabular_preprocessor.get_feature_names_out())
except Exception:
    main_tabular_feature_names = [f"feature_{i}" for i in range(X_train_tab.shape[1])]

# Raw lag features for XGBoost. Tree models do not require scaling.
X_train_hist = X_hist[train_idx]
X_test_hist = X_hist[test_idx]

y_train_tab = y_tab[train_idx]
y_test_tab = y_tab[test_idx]

# Sequence and target scalers are fitted on the outer-training set only.
scaler_x = StandardScaler()
scaler_y = StandardScaler()
scaler_x.fit(X_seq[train_idx].reshape(-1, 1))
scaler_y.fit(y_seq[train_idx].reshape(-1, 1))

X_seq_scaled = scaler_x.transform(
    X_seq.reshape(-1, 1)
).reshape(X_seq.shape[0], sequence_length, 1)
y_seq_scaled = scaler_y.transform(y_seq.reshape(-1, 1)).flatten()

X_train_seq = X_seq_scaled[train_idx]
X_test_seq = X_seq_scaled[test_idx]
y_train_seq = y_seq_scaled[train_idx]
y_test_seq = y_seq_scaled[test_idx]
y_test_real = y_seq[test_idx].copy()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

print("Outer training samples:", len(train_idx))
print("Reserved test samples:", len(test_idx))
print("Processed X_train_tab:", X_train_tab.shape)
print("Processed X_test_tab:", X_test_tab.shape)
print("X_train_hist:", X_train_hist.shape)
print("X_train_seq:", X_train_seq.shape)
print("The reserved test set is untouched by preprocessing fitting and model selection.")


## Baseline models

The main comparison contains:

- Linear Regression on processed hotel/search/FRED tabular features.
- MLP on the same processed tabular features.
- **XGBoost history-only**, using exactly the same five historical prices as the LSTM, represented as five lag variables.
- **XGBoost–LSTM late fusion**, with its prediction weight selected on inner validation data only.
- LSTM history-only.
- Candidate and final MLP–LSTM multimodal models.


In [ ]:
# Reset TensorFlow state for reproducible LSTM training
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model_lstm = Sequential([
    LSTM(64, input_shape=(sequence_length, 1)),
    Dense(32, activation="relu"),
    Dense(1)
])

model_lstm.compile(optimizer="adam", loss="mse")
model_lstm.summary()

history_lstm = model_lstm.fit(
    X_train_seq,
    y_train_seq,
    epochs=10,
    batch_size=128,
    callbacks=[early_stop],
    validation_split=0.2,
    verbose=1,
    shuffle=False
)

pred_lstm_scaled = model_lstm.predict(X_test_seq).flatten()
pred_lstm = scaler_y.inverse_transform(pred_lstm_scaled.reshape(-1, 1)).flatten()

lstm_metrics = regression_metrics(y_test_real, pred_lstm)
rmse_lstm = lstm_metrics["RMSE"]
mae_lstm = lstm_metrics["MAE"]

print("LSTM metrics:", lstm_metrics)


In [ ]:
# Baseline 2: Linear Regression using tabular variables + FRED indexes
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train_tab, y_train_tab)
pred_lr = model_lr.predict(X_test_tab)

lr_metrics = regression_metrics(y_test_tab, pred_lr)
rmse_lr = lr_metrics["RMSE"]
mae_lr = lr_metrics["MAE"]

print("Linear Regression metrics:", lr_metrics)


In [ ]:
# Baseline 3: MLP using leakage-safe processed tabular variables + FRED indexes
from sklearn.neural_network import MLPRegressor

model_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.20,
    n_iter_no_change=20,
)
model_mlp.fit(X_train_tab, y_train_tab)
pred_mlp = model_mlp.predict(X_test_tab)

mlp_metrics = regression_metrics(y_test_tab, pred_mlp)
rmse_mlp = mlp_metrics["RMSE"]
mae_mlp = mlp_metrics["MAE"]

print("MLP metrics:", mlp_metrics)


In [ ]:
# Baseline 4: XGBoost history-only
#
# To follow the supervisor's requested comparison, XGBoost receives exactly the
# same five historical prices as the LSTM. XGBoost sees them as five lag
# variables, whereas the LSTM receives them as an ordered sequence.
from xgboost import XGBRegressor

XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "n_jobs": -1,
    "objective": "reg:squarederror",
}

model_xgb = XGBRegressor(**XGB_PARAMS)
model_xgb.fit(X_train_hist, y_train_tab)
pred_xgb = model_xgb.predict(X_test_hist)

xgb_metrics = regression_metrics(y_test_tab, pred_xgb)
rmse_xgb = xgb_metrics["RMSE"]
mae_xgb = xgb_metrics["MAE"]

print("XGBoost history-only metrics:", xgb_metrics)


In [ ]:
# Baseline 5: XGBoost–LSTM late-fusion hybrid
#
# Both base learners use the same five historical prices but model them in
# different ways. The XGBoost weight is selected by repeated INNER validation;
# the reserved test set is never used to choose the fusion weight.

HYBRID_REPEATS = 3
HYBRID_VALIDATION_SIZE = 0.20
HYBRID_MAX_EPOCHS = 20
HYBRID_ALPHA_GRID = np.linspace(0.0, 1.0, 21)

hybrid_validation_rows = []

for repeat_id in range(1, HYBRID_REPEATS + 1):
    split_seed = SEED + repeat_id - 1
    inner_train_idx, inner_val_idx = train_test_split(
        np.asarray(train_idx),
        test_size=HYBRID_VALIDATION_SIZE,
        random_state=split_seed,
        shuffle=True,
    )

    # Inner XGBoost model on raw lag features.
    xgb_inner_params = XGB_PARAMS.copy()
    xgb_inner_params["random_state"] = split_seed
    model_xgb_inner = XGBRegressor(**xgb_inner_params)
    model_xgb_inner.fit(X_hist[inner_train_idx], y_seq[inner_train_idx])
    pred_val_xgb = model_xgb_inner.predict(X_hist[inner_val_idx])

    # Inner LSTM model. Sequence and target scalers are fitted on inner train only.
    seq_scaler_inner = StandardScaler()
    target_scaler_inner = StandardScaler()
    seq_scaler_inner.fit(X_seq[inner_train_idx].reshape(-1, 1))
    target_scaler_inner.fit(y_seq[inner_train_idx].reshape(-1, 1))

    X_inner_seq = seq_scaler_inner.transform(
        X_seq[inner_train_idx].reshape(-1, 1)
    ).reshape(len(inner_train_idx), sequence_length, 1)
    X_val_seq = seq_scaler_inner.transform(
        X_seq[inner_val_idx].reshape(-1, 1)
    ).reshape(len(inner_val_idx), sequence_length, 1)
    y_inner_scaled = target_scaler_inner.transform(
        y_seq[inner_train_idx].reshape(-1, 1)
    ).flatten()
    y_val_scaled = target_scaler_inner.transform(
        y_seq[inner_val_idx].reshape(-1, 1)
    ).flatten()

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(split_seed)
    model_lstm_inner = Sequential([
        Input(shape=(sequence_length, 1)),
        LSTM(64),
        Dense(32, activation="relu"),
        Dense(1),
    ])
    model_lstm_inner.compile(optimizer="adam", loss="mse")

    hybrid_early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        min_delta=1e-4,
        restore_best_weights=True,
    )
    model_lstm_inner.fit(
        X_inner_seq,
        y_inner_scaled,
        validation_data=(X_val_seq, y_val_scaled),
        epochs=HYBRID_MAX_EPOCHS,
        batch_size=128,
        callbacks=[hybrid_early_stop],
        verbose=0,
        shuffle=False,
    )

    pred_val_lstm_scaled = model_lstm_inner.predict(X_val_seq, verbose=0).flatten()
    pred_val_lstm = target_scaler_inner.inverse_transform(
        pred_val_lstm_scaled.reshape(-1, 1)
    ).flatten()

    for alpha_xgb in HYBRID_ALPHA_GRID:
        pred_val_hybrid = (
            alpha_xgb * pred_val_xgb
            + (1.0 - alpha_xgb) * pred_val_lstm
        )
        metrics = regression_metrics(y_seq[inner_val_idx], pred_val_hybrid)
        hybrid_validation_rows.append({
            "Repeat": repeat_id,
            "Split_Seed": split_seed,
            "Alpha_XGBoost": float(alpha_xgb),
            "Alpha_LSTM": float(1.0 - alpha_xgb),
            **metrics,
        })

hybrid_validation_df = pd.DataFrame(hybrid_validation_rows)
hybrid_alpha_summary_df = (
    hybrid_validation_df
    .groupby(["Alpha_XGBoost", "Alpha_LSTM"], as_index=False)
    .agg(
        Validation_RMSE_Mean=("RMSE", "mean"),
        Validation_RMSE_Std=("RMSE", "std"),
        Validation_MAE_Mean=("MAE", "mean"),
        Repeats=("Repeat", "nunique"),
    )
    .fillna({"Validation_RMSE_Std": 0.0})
    .sort_values(["Validation_RMSE_Mean", "Validation_RMSE_Std"])
    .reset_index(drop=True)
)

best_hybrid_row = hybrid_alpha_summary_df.iloc[0]
best_alpha_xgb = float(best_hybrid_row["Alpha_XGBoost"])
best_alpha_lstm = float(best_hybrid_row["Alpha_LSTM"])

# Combine predictions from the full outer-training XGBoost and LSTM models.
pred_xgb_lstm = best_alpha_xgb * pred_xgb + best_alpha_lstm * pred_lstm
xgb_lstm_metrics = regression_metrics(y_test_real, pred_xgb_lstm)
rmse_xgb_lstm = xgb_lstm_metrics["RMSE"]
mae_xgb_lstm = xgb_lstm_metrics["MAE"]

print("Validation-selected XGBoost weight:", best_alpha_xgb)
print("Validation-selected LSTM weight:", best_alpha_lstm)
print("XGBoost–LSTM validation summary:")
print(hybrid_alpha_summary_df.head(10))
print("XGBoost–LSTM TEST metrics:", xgb_lstm_metrics)

if SAVE_OUTPUT_CSV_FILES:
    hybrid_validation_df.to_csv("xgboost_lstm_validation_results.csv", index=False)
    hybrid_alpha_summary_df.to_csv("xgboost_lstm_weight_summary.csv", index=False)


## Candidate multimodal model: MLP + LSTM + FRED variables

This candidate model uses the expanded FRED set. Its tabular branch receives values produced by a preprocessing pipeline fitted on outer-training rows only, while the sequence branch receives the scaled five-month price history.


In [ ]:
# Reset TensorFlow state for reproducible multimodal training
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

# Candidate multimodal model: processed hotel/search/FRED features + history sequence.
# X_train_tab/X_test_tab were produced by a preprocessor fitted on outer train only.
input_tab = Input(shape=(X_train_tab.shape[1],), name="tabular_input")
x_tab = Dense(64, activation="relu")(input_tab)
x_tab = Dense(32, activation="relu")(x_tab)

input_seq = Input(shape=(sequence_length, 1), name="price_history_input")
x_seq = LSTM(32)(input_seq)

combined = Concatenate(name="feature_fusion")([x_tab, x_seq])
shortcut = Dense(32, use_bias=False, name="residual_projection")(combined)
x = Dense(32, activation="relu")(combined)
x = Add(name="residual_add")([x, shortcut])
output = Dense(1, name="price_output")(x)

model_fusion = Model(inputs=[input_tab, input_seq], outputs=output)
model_fusion.compile(optimizer="adam", loss="mse")
model_fusion.summary()

history_fusion = model_fusion.fit(
    [X_train_tab, X_train_seq],
    y_train_seq,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False,
)

pred_fusion_scaled = model_fusion.predict(
    [X_test_tab, X_test_seq], verbose=0
).flatten()
pred_fusion = scaler_y.inverse_transform(
    pred_fusion_scaled.reshape(-1, 1)
).flatten()

fusion_metrics = regression_metrics(y_test_real, pred_fusion)
rmse_mm_tab_time = fusion_metrics["RMSE"]
mae_mm_tab_time = fusion_metrics["MAE"]

print("Candidate fusion model metrics:", fusion_metrics)


In [ ]:
results_df = pd.DataFrame([
    {"Model": "Linear Regression", **lr_metrics},
    {"Model": "MLP", **mlp_metrics},
    {"Model": "XGBoost (History)", **xgb_metrics},
    {"Model": "XGBoost–LSTM", **xgb_lstm_metrics},
    {"Model": "LSTM", **lstm_metrics},
    {"Model": "Candidate MM(Expanded FRED)", **fusion_metrics},
]).sort_values("RMSE").reset_index(drop=True)

print(results_df)
if SAVE_OUTPUT_CSV_FILES:
    results_df.to_csv("main_model_comparison_results.csv", index=False)



&lt;h2 id="General-ablation-study"&gt;General ablation study<a class="anchor-link" href="#General-ablation-study">¶</a>&lt;/h2&gt;&lt;p&gt;This ablation is now evaluated with a branch-matched neural model rather than Ridge and Random Forest:&lt;/p&gt;
<ul>
<li>history-only settings use the LSTM branch;</li>
<li>hotel/search-only settings use the MLP branch;</li>
<li>combined settings use the same MLP–LSTM fusion structure as the final model.</li>
</ul>
&lt;p&gt;Every feature set is scored on repeated validation splits drawn only from the outer training set. The reserved test set is not used.&lt;/p&gt;



In [ ]:
# Repeated validation makes the feature-set comparison less dependent on one
# lucky split. Increase these values for a more expensive robustness check.
ABLATION_REPEATS = 3
ABLATION_VALIDATION_SIZE = 0.20
ABLATION_MAX_EPOCHS = 20

# A fixed reference architecture is used for ALL ablation feature sets so that
# the feature comparison is not confounded by changing model capacity.
ABLATION_MODEL_PARAMS = {
    "lstm_units": 32,
    "tab_dense_1": 64,
    "tab_dense_2": 32,
    "fusion_dense": 32,
    "dropout": 0.0,
    "learning_rate": 1e-3,
    "batch_size": 128,
}


def make_dense_onehot_encoder():
    """Return a dense OneHotEncoder compatible with old and new sklearn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_tabular_preprocessor(X_raw):
    """Create a preprocessing pipeline that is fitted on inner-training data only."""
    cat_cols = [
        c for c in X_raw.columns
        if X_raw[c].dtype == "object" or str(X_raw[c].dtype) == "category"
    ]
    num_cols = [c for c in X_raw.columns if c not in cat_cols]

    transformers = []
    if num_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ))
    if cat_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_dense_onehot_encoder())
            ]),
            cat_cols
        ))

    if not transformers:
        return None

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_branch_matched_ablation_model(
    input_tab_dim,
    use_sequence,
    params=ABLATION_MODEL_PARAMS
):
    """
    Build the branch structure that matches the selected modalities.

    - tabular only  -> MLP
    - sequence only -> LSTM
    - both          -> MLP-LSTM fusion
    """
    model_inputs = []
    branches = []

    if input_tab_dim > 0:
        input_tab = Input(shape=(input_tab_dim,), name="tabular_input")
        x_tab = Dense(params["tab_dense_1"], activation="relu")(input_tab)
        if params["dropout"] > 0:
            x_tab = Dropout(params["dropout"])(x_tab)
        x_tab = Dense(params["tab_dense_2"], activation="relu")(x_tab)
        model_inputs.append(input_tab)
        branches.append(x_tab)

    if use_sequence:
        input_seq = Input(
            shape=(sequence_length, 1),
            name="price_history_input"
        )
        x_seq = LSTM(params["lstm_units"])(input_seq)
        model_inputs.append(input_seq)
        branches.append(x_seq)

    if not branches:
        raise ValueError("An ablation model must contain at least one input branch.")

    if len(branches) == 2:
        combined = Concatenate(name="feature_fusion")(branches)
    else:
        combined = branches[0]

    shortcut = Dense(
        params["fusion_dense"],
        use_bias=False,
        name="residual_projection"
    )(combined)
    x = Dense(params["fusion_dense"], activation="relu")(combined)
    if params["dropout"] > 0:
        x = Dropout(params["dropout"])(x)
    x = Add(name="residual_add")([x, shortcut])
    output = Dense(1, name="price_output")(x)

    final_inputs = model_inputs[0] if len(model_inputs) == 1 else model_inputs
    model = Model(inputs=final_inputs, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=params["learning_rate"]),
        loss="mse"
    )
    return model


def pack_ablation_inputs(tab_values, seq_values, use_tabular, use_sequence):
    """Use an array for one branch and a list for two branches."""
    arrays = []
    if use_tabular:
        arrays.append(tab_values)
    if use_sequence:
        arrays.append(seq_values)
    return arrays[0] if len(arrays) == 1 else arrays


def evaluate_neural_ablation(
    feature_set_name,
    selected_cols,
    repeats=ABLATION_REPEATS
):
    """
    Score one feature set using repeated INNER validation only.

    The function never reads test_idx or y_test. Each repeat:
      1. splits the outer training set into inner-train and validation;
      2. fits preprocessing/scalers on inner-train only;
      3. trains the matching neural branch structure;
      4. evaluates in original price units on validation.
    """
    selected_cols = [c for c in selected_cols if c in aligned_df.columns]
    hist_price_cols_local = [
        f"hist_price_{i + 1}" for i in range(sequence_length)
    ]

    use_sequence = any(c in selected_cols for c in hist_price_cols_local)
    selected_tabular_cols = [
        c for c in selected_cols if c not in hist_price_cols_local
    ]
    use_tabular = len(selected_tabular_cols) > 0

    X_tab_raw_local = (
        aligned_df[selected_tabular_cols].copy()
        if use_tabular else None
    )

    repeat_rows = []

    for repeat_id in range(1, repeats + 1):
        split_seed = SEED + repeat_id - 1
        inner_train_idx, inner_val_idx = train_test_split(
            np.asarray(train_idx),
            test_size=ABLATION_VALIDATION_SIZE,
            random_state=split_seed,
            shuffle=True
        )

        # Tabular preprocessing is fitted on inner-training data only.
        if use_tabular:
            preprocessor = build_tabular_preprocessor(X_tab_raw_local)
            X_inner_tab = preprocessor.fit_transform(
                X_tab_raw_local.iloc[inner_train_idx]
            )
            X_val_tab = preprocessor.transform(
                X_tab_raw_local.iloc[inner_val_idx]
            )
            X_inner_tab = np.asarray(X_inner_tab, dtype=np.float32)
            X_val_tab = np.asarray(X_val_tab, dtype=np.float32)
            input_tab_dim = X_inner_tab.shape[1]
        else:
            X_inner_tab = None
            X_val_tab = None
            input_tab_dim = 0

        # Sequence and target scalers are also fitted on inner-training only.
        if use_sequence:
            seq_scaler = StandardScaler()
            seq_scaler.fit(X_seq[inner_train_idx].reshape(-1, 1))
            X_inner_seq_local = seq_scaler.transform(
                X_seq[inner_train_idx].reshape(-1, 1)
            ).reshape(len(inner_train_idx), sequence_length, 1)
            X_val_seq_local = seq_scaler.transform(
                X_seq[inner_val_idx].reshape(-1, 1)
            ).reshape(len(inner_val_idx), sequence_length, 1)
        else:
            X_inner_seq_local = None
            X_val_seq_local = None

        target_scaler = StandardScaler()
        target_scaler.fit(y_seq[inner_train_idx].reshape(-1, 1))
        y_inner_scaled = target_scaler.transform(
            y_seq[inner_train_idx].reshape(-1, 1)
        ).flatten()
        y_val_scaled = target_scaler.transform(
            y_seq[inner_val_idx].reshape(-1, 1)
        ).flatten()

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(split_seed)

        model = build_branch_matched_ablation_model(
            input_tab_dim=input_tab_dim,
            use_sequence=use_sequence,
            params=ABLATION_MODEL_PARAMS
        )

        early_stop_local = EarlyStopping(
            monitor="val_loss",
            patience=3,
            min_delta=1e-4,
            restore_best_weights=True
        )

        train_inputs = pack_ablation_inputs(
            X_inner_tab,
            X_inner_seq_local,
            use_tabular,
            use_sequence
        )
        val_inputs = pack_ablation_inputs(
            X_val_tab,
            X_val_seq_local,
            use_tabular,
            use_sequence
        )

        history = model.fit(
            train_inputs,
            y_inner_scaled,
            validation_data=(val_inputs, y_val_scaled),
            epochs=ABLATION_MAX_EPOCHS,
            batch_size=ABLATION_MODEL_PARAMS["batch_size"],
            callbacks=[early_stop_local],
            verbose=0,
            shuffle=False
        )

        pred_val_scaled = model.predict(val_inputs, verbose=0).flatten()
        pred_val = target_scaler.inverse_transform(
            pred_val_scaled.reshape(-1, 1)
        ).flatten()
        metrics = regression_metrics(y_seq[inner_val_idx], pred_val)

        repeat_rows.append({
            "Feature_Set": feature_set_name,
            "Repeat": repeat_id,
            "Split_Seed": split_seed,
            "Branch_Type": (
                "MLP-LSTM"
                if use_tabular and use_sequence
                else "MLP"
                if use_tabular
                else "LSTM"
            ),
            "Num_Features_Before_Encoding": len(selected_cols),
            "Best_Epoch": int(np.argmin(history.history["val_loss"]) + 1),
            **metrics
        })

    repeat_df = pd.DataFrame(repeat_rows)

    summary = {
        "Feature_Set": feature_set_name,
        "Model": "Branch-matched neural scorer",
        "Branch_Type": repeat_df["Branch_Type"].iloc[0],
        "Num_Features_Before_Encoding": len(selected_cols),
        "Num_Repeats": repeats,
        "RMSE": repeat_df["RMSE"].mean(),
        "RMSE_Std": repeat_df["RMSE"].std(ddof=1) if repeats > 1 else 0.0,
        "MAE": repeat_df["MAE"].mean(),
        "MAE_Std": repeat_df["MAE"].std(ddof=1) if repeats > 1 else 0.0,
        "MAPE": repeat_df["MAPE"].mean(),
        "MAPE_Std": repeat_df["MAPE"].std(ddof=1) if repeats > 1 else 0.0,
        "R2": repeat_df["R2"].mean(),
        "R2_Std": repeat_df["R2"].std(ddof=1) if repeats > 1 else 0.0,
        "Median_Best_Epoch": int(round(repeat_df["Best_Epoch"].median()))
    }
    return summary, repeat_df


hist_price_cols = [
    f"hist_price_{i + 1}" for i in range(sequence_length)
]
hotel_search_cols = base_tabular_cols
macro_index_cols = exogenous_cols

ablation_specs = [
    ("History only", hist_price_cols),
    ("Hotel/search only", hotel_search_cols),
    ("Hotel/search + FRED indexes", hotel_search_cols + macro_index_cols),
    ("History + hotel/search", hist_price_cols + hotel_search_cols),
    (
        "History + hotel/search + FRED indexes",
        hist_price_cols + hotel_search_cols + macro_index_cols
    ),
]

ablation_summaries = []
ablation_repeat_frames = []

for feature_set_name, cols in ablation_specs:
    print("Evaluating:", feature_set_name)
    summary, repeat_df = evaluate_neural_ablation(
        feature_set_name,
        cols,
        repeats=ABLATION_REPEATS
    )
    ablation_summaries.append(summary)
    ablation_repeat_frames.append(repeat_df)

ablation_df = (
    pd.DataFrame(ablation_summaries)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
ablation_repeat_df = pd.concat(
    ablation_repeat_frames,
    ignore_index=True
)

print("\nGeneral neural ablation summary (validation only):")
print(ablation_df)
print("\nPer-repeat general ablation results:")
print(ablation_repeat_df)

if SAVE_OUTPUT_CSV_FILES:
    ablation_df.to_csv("general_ablation_results.csv", index=False)
    ablation_repeat_df.to_csv(
        "general_ablation_repeat_results.csv",
        index=False
    )


# -------------------------------------------------------------------------
# General ablation figure: mean validation RMSE with repeat-to-repeat error bars
# -------------------------------------------------------------------------
general_ablation_plot_df = ablation_df.sort_values("RMSE", ascending=True).copy()
fig_height = max(5.5, 0.65 * len(general_ablation_plot_df))
fig, ax = plt.subplots(figsize=(10, fig_height))
ax.barh(
    general_ablation_plot_df["Feature_Set"],
    general_ablation_plot_df["RMSE"],
    xerr=general_ablation_plot_df["RMSE_Std"],
    capsize=4,
)
ax.invert_yaxis()
ax.set_xlabel("Mean validation RMSE")
ax.set_ylabel("Feature set")
ax.set_title("General Ablation Study: Mean Validation RMSE ± 1 SD")
ax.grid(axis="x", alpha=0.25)
for y_pos, value in enumerate(general_ablation_plot_df["RMSE"]):
    ax.text(value, y_pos, f"  {value:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()
plt.close(fig)



&lt;h2 id="Expanded-index-ablation-study"&gt;Expanded index ablation study<a class="anchor-link" href="#Expanded-index-ablation-study">¶</a>&lt;/h2&gt;&lt;p&gt;The FRED feature-set comparison now uses the same MLP–LSTM fusion logic as the final model. Each setting is evaluated over repeated inner validation splits from the outer training set.&lt;/p&gt;
&lt;p&gt;The held-out test set is not used to decide which FRED columns are retained.&lt;/p&gt;



In [ ]:
baseline_cols = hist_price_cols + hotel_search_cols

# Index-only feature pool: one normalized base-100 index for each loaded FRED series.
all_loaded_fred_index_cols = [
    c for c in fred_index_cols if c in aligned_df.columns
]

# Full FRED feature pool: index + MoM/YoY changes.
all_loaded_fred_feature_cols = [
    c for c in fred_all_feature_cols if c in aligned_df.columns
]

# Main-model FRED feature set: all normalized indexes plus core MoM/YoY.
main_model_fred_cols = [
    c for c in exogenous_cols if c in aligned_df.columns
]

print("Number of loaded FRED index columns:", len(all_loaded_fred_index_cols))
print(
    "Number of loaded FRED feature columns including MoM/YoY:",
    len(all_loaded_fred_feature_cols)
)
print("Number of main-model FRED columns:", len(main_model_fred_cols))

index_ablation_specs = []

# 1. Baseline without FRED.
index_ablation_specs.append((
    "Baseline: no FRED index",
    baseline_cols
))

# 2. Supervisor-style expanded index pool.
index_ablation_specs.append((
    "Baseline + all loaded FRED index values",
    baseline_cols + all_loaded_fred_index_cols
))

# 3. Main model feature version.
index_ablation_specs.append((
    "Baseline + main FRED features",
    baseline_cols + main_model_fred_cols
))

# 4. Full pool including all MoM/YoY features.
index_ablation_specs.append((
    "Baseline + all FRED index/MoM/YoY features",
    baseline_cols + all_loaded_fred_feature_cols
))

# 5. Add one macroeconomic category at a time.
for category, cols in fred_category_index_groups.items():
    cols = [c for c in cols if c in aligned_df.columns]
    if cols:
        index_ablation_specs.append((
            f"Baseline + category: {category}",
            baseline_cols + cols
        ))

# 6. Leave one macroeconomic category out from the full index-value pool.
for category, cols in fred_category_index_groups.items():
    cols = [c for c in cols if c in aligned_df.columns]
    if cols:
        remaining = [
            c for c in all_loaded_fred_index_cols if c not in cols
        ]
        index_ablation_specs.append((
            f"All FRED indexes - category: {category}",
            baseline_cols + remaining
        ))

# 7. Core individual indicators with their index/MoM/YoY features.
for key in ["inflation_cpi", "fuel_price", "electricity"]:
    cols = [
        c for c in fred_indicator_feature_groups.get(key, [])
        if c in aligned_df.columns
    ]
    if cols:
        index_ablation_specs.append((
            f"Baseline + core indicator: {key}",
            baseline_cols + cols
        ))

# Remove duplicate feature-set names while preserving order.
seen_specs = set()
deduped_specs = []
for feature_set_name, cols in index_ablation_specs:
    if feature_set_name not in seen_specs:
        deduped_specs.append((feature_set_name, cols))
        seen_specs.add(feature_set_name)

index_ablation_summaries = []
index_ablation_repeat_frames = []

for feature_set_name, cols in deduped_specs:
    print("Evaluating FRED feature set:", feature_set_name)
    summary, repeat_df = evaluate_neural_ablation(
        feature_set_name,
        cols,
        repeats=ABLATION_REPEATS
    )
    index_ablation_summaries.append(summary)
    index_ablation_repeat_frames.append(repeat_df)

index_ablation_df = (
    pd.DataFrame(index_ablation_summaries)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
index_ablation_repeat_df = pd.concat(
    index_ablation_repeat_frames,
    ignore_index=True
)

# Calculate improvement against the no-FRED validation baseline.
baseline_row = index_ablation_df[
    index_ablation_df["Feature_Set"] == "Baseline: no FRED index"
].iloc[0]

baseline_rmse = float(baseline_row["RMSE"])
baseline_rmse_std = float(baseline_row["RMSE_Std"])

index_ablation_compare_df = index_ablation_df.copy()
index_ablation_compare_df["Baseline_RMSE"] = baseline_rmse
index_ablation_compare_df["Baseline_RMSE_Std"] = baseline_rmse_std
index_ablation_compare_df["RMSE_Improvement_vs_Baseline"] = (
    baseline_rmse - index_ablation_compare_df["RMSE"]
)
index_ablation_compare_df["Improvement_%"] = (
    index_ablation_compare_df["RMSE_Improvement_vs_Baseline"]
    / baseline_rmse
    * 100
)

print("\nExpanded neural index ablation results (validation only):")
print(index_ablation_compare_df)
print("\nPer-repeat expanded index ablation results:")
print(index_ablation_repeat_df)

if SAVE_OUTPUT_CSV_FILES:
    index_ablation_compare_df.to_csv(
        "expanded_index_ablation_results.csv",
        index=False
    )
    index_ablation_repeat_df.to_csv(
        "expanded_index_ablation_repeat_results.csv",
        index=False
    )

# Optional descriptive helper. This table is NOT used for feature selection.
# Correlations are calculated on the outer training rows only, so the reserved
# test target is not inspected.
fred_corr_rows = []
for col in all_loaded_fred_feature_cols:
    if col in aligned_df.columns:
        corr_data = aligned_df.iloc[train_idx][
            [col, "target_price"]
        ].apply(pd.to_numeric, errors="coerce")
        corr = corr_data.corr().iloc[0, 1]
        fred_corr_rows.append({
            "FRED_Feature": col,
            "Abs_Correlation_With_Target": (
                abs(corr) if pd.notna(corr) else np.nan
            ),
            "Correlation_With_Target": corr
        })

fred_feature_screening_df = (
    pd.DataFrame(fred_corr_rows)
    .sort_values("Abs_Correlation_With_Target", ascending=False)
    .reset_index(drop=True)
)

print("\nTop FRED features by absolute training-set correlation:")
print(fred_feature_screening_df.head(30))

if SAVE_OUTPUT_CSV_FILES:
    fred_feature_screening_df.to_csv(
        "fred_feature_screening_top30.csv",
        index=False
    )


# -------------------------------------------------------------------------
# FRED ablation figures
# -------------------------------------------------------------------------
# 1. Mean RMSE for every FRED configuration, with repeated-validation SD.
fred_rmse_plot_df = index_ablation_compare_df.sort_values("RMSE", ascending=True).copy()
fig_height = max(7.0, 0.42 * len(fred_rmse_plot_df))
fig, ax = plt.subplots(figsize=(12, fig_height))
ax.barh(
    fred_rmse_plot_df["Feature_Set"],
    fred_rmse_plot_df["RMSE"],
    xerr=fred_rmse_plot_df["RMSE_Std"],
    capsize=3,
)
ax.invert_yaxis()
ax.axvline(baseline_rmse, linestyle="--", linewidth=1.2, label="No-FRED baseline")
ax.set_xlabel("Mean validation RMSE")
ax.set_ylabel("FRED feature configuration")
ax.set_title("Expanded FRED Ablation: Mean Validation RMSE ± 1 SD")
ax.grid(axis="x", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

# 2. Improvement relative to the no-FRED baseline.
fred_improvement_plot_df = index_ablation_compare_df.sort_values(
    "RMSE_Improvement_vs_Baseline", ascending=True
).copy()
fig_height = max(7.0, 0.42 * len(fred_improvement_plot_df))
fig, ax = plt.subplots(figsize=(12, fig_height))
ax.barh(
    fred_improvement_plot_df["Feature_Set"],
    fred_improvement_plot_df["RMSE_Improvement_vs_Baseline"],
)
ax.axvline(0, linestyle="--", linewidth=1.2)
ax.set_xlabel("RMSE improvement versus no-FRED baseline")
ax.set_ylabel("FRED feature configuration")
ax.set_title("Expanded FRED Ablation: Positive Values Indicate Improvement")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()
plt.close(fig)



&lt;h2 id="Ablation-selected-final-FRED-feature-subset"&gt;Ablation-selected final FRED feature subset<a class="anchor-link" href="#Ablation-selected-final-FRED-feature-subset">¶</a>&lt;/h2&gt;&lt;p&gt;The selected FRED subset is based on mean repeated-validation RMSE from the neural MLP–LSTM ablation scorer. No test-set result is used in this decision.&lt;/p&gt;



In [ ]:
# Ablation-selected final FRED feature subset

feature_set_to_cols = {
    feature_set_name: [c for c in cols if c in aligned_df.columns]
    for feature_set_name, cols in deduped_specs
}

fred_feature_pool_set = set(
    all_loaded_fred_index_cols
    + all_loaded_fred_feature_cols
    + main_model_fred_cols
)

# One summary row now represents the mean over repeated validation runs.
ablation_selection_summary_df = (
    index_ablation_compare_df[
        [
            "Feature_Set",
            "RMSE",
            "RMSE_Std",
            "MAE",
            "R2",
            "RMSE_Improvement_vs_Baseline",
            "Improvement_%",
            "Num_Repeats"
        ]
    ]
    .rename(columns={
        "RMSE": "Mean_RMSE",
        "RMSE_Std": "Std_RMSE",
        "MAE": "Mean_MAE",
        "R2": "Mean_R2",
        "RMSE_Improvement_vs_Baseline": "Mean_RMSE_Improvement",
        "Improvement_%": "Mean_Improvement_Percent",
        "Num_Repeats": "Validation_Repeats"
    })
    .copy()
)

ablation_selection_summary_df["Has_FRED_Features"] = (
    ablation_selection_summary_df["Feature_Set"].apply(
        lambda name: any(
            c in fred_feature_pool_set
            for c in feature_set_to_cols.get(name, [])
        )
    )
)

ablation_selection_summary_df = (
    ablation_selection_summary_df
    .sort_values(["Mean_RMSE", "Std_RMSE"])
    .reset_index(drop=True)
)

print("Ablation feature-set ranking used for final FRED selection:")
print(ablation_selection_summary_df)

# Prefer a FRED feature set only when its mean repeated-validation RMSE is
# lower than the no-FRED baseline. The test set is not involved.
positive_fred_candidates = ablation_selection_summary_df[
    (ablation_selection_summary_df["Has_FRED_Features"] == True)
    & (ablation_selection_summary_df["Mean_RMSE_Improvement"] > 0)
].copy()

if len(positive_fred_candidates) > 0:
    selected_ablation_feature_set = (
        positive_fred_candidates.iloc[0]["Feature_Set"]
    )
    selection_reason = (
        "Lowest mean repeated-validation RMSE among FRED settings that "
        "improved on the no-FRED validation baseline."
    )
else:
    selected_ablation_feature_set = "Baseline: no FRED index"
    selection_reason = (
        "No FRED setting improved the no-FRED baseline on mean repeated "
        "validation RMSE; the final model therefore uses no FRED variables."
    )

selected_ablation_cols = feature_set_to_cols.get(
    selected_ablation_feature_set,
    []
)
selected_fred_cols = [
    c for c in selected_ablation_cols if c in fred_feature_pool_set
]

# Keep order, remove duplicates, and retain only available columns.
selected_fred_cols = list(dict.fromkeys(selected_fred_cols))
selected_fred_cols = [
    c for c in selected_fred_cols if c in aligned_df.columns
]

print("\nSelected ablation feature set:", selected_ablation_feature_set)
print("Selection reason:", selection_reason)
print("Selected FRED columns for the final multimodal model:")
print(selected_fred_cols)
print("Number of selected FRED columns:", len(selected_fred_cols))

# The final tabular branch uses hotel/search/date variables plus selected FRED.
optimized_tabular_cols = base_tabular_cols + selected_fred_cols
optimized_tabular_cols = [
    c for c in optimized_tabular_cols if c in aligned_df.columns
]

X_tab_final_raw = aligned_df[optimized_tabular_cols].copy()
X_tab_final_raw = X_tab_final_raw.replace([np.inf, -np.inf], np.nan)

# Fit the complete final categorical vocabulary, imputers and numeric scaler on
# outer-training rows only. The reserved test rows are transform-only.
final_tabular_preprocessor = build_tabular_preprocessor(X_tab_final_raw)
X_train_tab_final = final_tabular_preprocessor.fit_transform(
    X_tab_final_raw.iloc[train_idx]
)
X_test_tab_final = final_tabular_preprocessor.transform(
    X_tab_final_raw.iloc[test_idx]
)
X_train_tab_final = np.asarray(X_train_tab_final, dtype=np.float32)
X_test_tab_final = np.asarray(X_test_tab_final, dtype=np.float32)

try:
    final_tabular_feature_names = list(
        final_tabular_preprocessor.get_feature_names_out()
    )
except Exception:
    final_tabular_feature_names = [
        f"feature_{i}" for i in range(X_train_tab_final.shape[1])
    ]

print("Final raw tabular matrix shape:", X_tab_final_raw.shape)
print("Final processed train tabular shape:", X_train_tab_final.shape)
print("Final processed test tabular shape:", X_test_tab_final.shape)
print("Final processed feature count:", len(final_tabular_feature_names))



&lt;h2 id="Hyperparameter-experiment"&gt;Hyperparameter experiment<a class="anchor-link" href="#Hyperparameter-experiment">¶</a>&lt;/h2&gt;&lt;p&gt;Hyperparameters are selected by <strong>mean validation RMSE across repeated inner splits</strong> of the outer training set.&lt;/p&gt;
&lt;p&gt;For every repeat, tabular, sequence, and target scalers are fitted only on the inner-training partition. The reserved test set is never predicted during tuning.&lt;/p&gt;



In [ ]:
# Repeated-validation hyperparameter experiment for the ablation-selected
# multimodal model. The test set is not used anywhere in this cell.
import tensorflow as tf

HYPERPARAMETER_REPEATS = 3
TUNING_VALIDATION_SIZE = 0.20
TUNING_MAX_EPOCHS = 30
TUNING_PATIENCE = 5

# A broader but still practical grid. All settings are compared on exactly the
# same repeated validation splits.
hyperparameter_grid = [
    {
        "lstm_units": 32,
        "tab_dense_1": 64,
        "tab_dense_2": 32,
        "fusion_dense": 32,
        "dropout": 0.0,
        "learning_rate": 1e-3,
        "batch_size": 128
    },
    {
        "lstm_units": 64,
        "tab_dense_1": 128,
        "tab_dense_2": 64,
        "fusion_dense": 64,
        "dropout": 0.0,
        "learning_rate": 1e-3,
        "batch_size": 128
    },
    {
        "lstm_units": 64,
        "tab_dense_1": 128,
        "tab_dense_2": 64,
        "fusion_dense": 64,
        "dropout": 0.1,
        "learning_rate": 1e-3,
        "batch_size": 128
    },
    {
        "lstm_units": 64,
        "tab_dense_1": 128,
        "tab_dense_2": 64,
        "fusion_dense": 64,
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 128
    },
    {
        "lstm_units": 32,
        "tab_dense_1": 128,
        "tab_dense_2": 64,
        "fusion_dense": 64,
        "dropout": 0.1,
        "learning_rate": 5e-4,
        "batch_size": 64
    },
    {
        "lstm_units": 64,
        "tab_dense_1": 256,
        "tab_dense_2": 128,
        "fusion_dense": 64,
        "dropout": 0.1,
        "learning_rate": 5e-4,
        "batch_size": 128
    },
    {
        "lstm_units": 128,
        "tab_dense_1": 128,
        "tab_dense_2": 64,
        "fusion_dense": 64,
        "dropout": 0.1,
        "learning_rate": 5e-4,
        "batch_size": 128
    },
    {
        "lstm_units": 64,
        "tab_dense_1": 128,
        "tab_dense_2": 64,
        "fusion_dense": 128,
        "dropout": 0.1,
        "learning_rate": 1e-3,
        "batch_size": 64
    },
]


def build_multimodal_model(input_tab_dim, params):
    # Tabular branch
    input_tab = Input(
        shape=(input_tab_dim,),
        name="tabular_input"
    )
    x_tab = Dense(
        params["tab_dense_1"],
        activation="relu"
    )(input_tab)
    if params["dropout"] > 0:
        x_tab = Dropout(params["dropout"])(x_tab)
    x_tab = Dense(
        params["tab_dense_2"],
        activation="relu"
    )(x_tab)

    # Time-series branch
    input_seq = Input(
        shape=(sequence_length, 1),
        name="price_history_input"
    )
    x_seq = LSTM(params["lstm_units"])(input_seq)

    # Fusion branch with residual projection
    combined = Concatenate(name="feature_fusion")([x_tab, x_seq])
    shortcut = Dense(
        params["fusion_dense"],
        use_bias=False,
        name="residual_projection"
    )(combined)
    x = Dense(
        params["fusion_dense"],
        activation="relu"
    )(combined)
    if params["dropout"] > 0:
        x = Dropout(params["dropout"])(x)
    x = Add(name="residual_add")([x, shortcut])
    output = Dense(1, name="price_output")(x)

    model = Model(
        inputs=[input_tab, input_seq],
        outputs=output
    )
    model.compile(
        optimizer=Adam(
            learning_rate=params["learning_rate"]
        ),
        loss="mse"
    )
    return model


hyperparameter_repeat_rows = []

for run_id, params in enumerate(hyperparameter_grid, start=1):
    print(
        f"\nHyperparameter setting {run_id}/"
        f"{len(hyperparameter_grid)}: {params}"
    )

    for repeat_id in range(1, HYPERPARAMETER_REPEATS + 1):
        split_seed = SEED + repeat_id - 1
        inner_train_idx, inner_val_idx = train_test_split(
            np.asarray(train_idx),
            test_size=TUNING_VALIDATION_SIZE,
            random_state=split_seed,
            shuffle=True
        )

        # Repeat-local tabular preprocessing: imputation, scaling and
        # one-hot encoding are fitted only on inner training.
        tab_preprocessor_hp = build_tabular_preprocessor(
            X_tab_final_raw
        )
        X_inner_tab_hp = tab_preprocessor_hp.fit_transform(
            X_tab_final_raw.iloc[inner_train_idx]
        )
        X_val_tab_hp = tab_preprocessor_hp.transform(
            X_tab_final_raw.iloc[inner_val_idx]
        )
        X_inner_tab_hp = np.asarray(
            X_inner_tab_hp,
            dtype=np.float32
        )
        X_val_tab_hp = np.asarray(
            X_val_tab_hp,
            dtype=np.float32
        )

        seq_scaler_hp = StandardScaler()
        seq_scaler_hp.fit(
            X_seq[inner_train_idx].reshape(-1, 1)
        )
        X_inner_seq_hp = seq_scaler_hp.transform(
            X_seq[inner_train_idx].reshape(-1, 1)
        ).reshape(len(inner_train_idx), sequence_length, 1)
        X_val_seq_hp = seq_scaler_hp.transform(
            X_seq[inner_val_idx].reshape(-1, 1)
        ).reshape(len(inner_val_idx), sequence_length, 1)

        target_scaler_hp = StandardScaler()
        target_scaler_hp.fit(
            y_seq[inner_train_idx].reshape(-1, 1)
        )
        y_inner_hp = target_scaler_hp.transform(
            y_seq[inner_train_idx].reshape(-1, 1)
        ).flatten()
        y_val_hp = target_scaler_hp.transform(
            y_seq[inner_val_idx].reshape(-1, 1)
        ).flatten()

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(split_seed)

        model_hp = build_multimodal_model(
            input_tab_dim=X_inner_tab_hp.shape[1],
            params=params
        )

        early_stop_hp = EarlyStopping(
            monitor="val_loss",
            patience=TUNING_PATIENCE,
            min_delta=1e-4,
            restore_best_weights=True
        )

        history_hp = model_hp.fit(
            [X_inner_tab_hp, X_inner_seq_hp],
            y_inner_hp,
            validation_data=(
                [X_val_tab_hp, X_val_seq_hp],
                y_val_hp
            ),
            epochs=TUNING_MAX_EPOCHS,
            batch_size=params["batch_size"],
            callbacks=[early_stop_hp],
            verbose=0,
            shuffle=False
        )

        pred_val_scaled = model_hp.predict(
            [X_val_tab_hp, X_val_seq_hp],
            verbose=0
        ).flatten()
        pred_val = target_scaler_hp.inverse_transform(
            pred_val_scaled.reshape(-1, 1)
        ).flatten()
        metrics_hp = regression_metrics(
            y_seq[inner_val_idx],
            pred_val
        )

        hyperparameter_repeat_rows.append({
            "Run": run_id,
            "Repeat": repeat_id,
            "Split_Seed": split_seed,
            "Sequence_Length": sequence_length,
            "Prediction_Horizon": prediction_horizon,
            "LSTM_Units": params["lstm_units"],
            "Tab_Dense_1": params["tab_dense_1"],
            "Tab_Dense_2": params["tab_dense_2"],
            "Fusion_Dense": params["fusion_dense"],
            "Dropout": params["dropout"],
            "Learning_Rate": params["learning_rate"],
            "Batch_Size": params["batch_size"],
            "Best_Val_Loss_Scaled": min(
                history_hp.history["val_loss"]
            ),
            "Best_Epoch": int(
                np.argmin(history_hp.history["val_loss"]) + 1
            ),
            "Epochs_Trained": len(
                history_hp.history["loss"]
            ),
            **metrics_hp
        })

hyperparameter_repeat_df = pd.DataFrame(
    hyperparameter_repeat_rows
)

parameter_columns = [
    "Run",
    "Sequence_Length",
    "Prediction_Horizon",
    "LSTM_Units",
    "Tab_Dense_1",
    "Tab_Dense_2",
    "Fusion_Dense",
    "Dropout",
    "Learning_Rate",
    "Batch_Size"
]

hyperparameter_df = (
    hyperparameter_repeat_df
    .groupby(parameter_columns, as_index=False)
    .agg(
        Validation_RMSE_Mean=("RMSE", "mean"),
        Validation_RMSE_Std=("RMSE", "std"),
        Validation_MAE_Mean=("MAE", "mean"),
        Validation_MAPE_Mean=("MAPE", "mean"),
        Validation_R2_Mean=("R2", "mean"),
        Mean_Best_Epoch=("Best_Epoch", "mean"),
        Median_Best_Epoch=("Best_Epoch", "median"),
        Mean_Epochs_Trained=("Epochs_Trained", "mean"),
        Repeats=("Repeat", "nunique")
    )
)

hyperparameter_df["Validation_RMSE_Std"] = (
    hyperparameter_df["Validation_RMSE_Std"].fillna(0.0)
)
hyperparameter_df = (
    hyperparameter_df
    .sort_values([
        "Validation_RMSE_Mean",
        "Validation_RMSE_Std"
    ])
    .reset_index(drop=True)
)

print("\nHyperparameter summary ranked by repeated VALIDATION RMSE:")
print(hyperparameter_df)

print("\nPer-repeat hyperparameter results:")
print(hyperparameter_repeat_df)

best_hyperparameters = hyperparameter_df.iloc[0]
print("\nSelected hyperparameters (test set still untouched):")
print(best_hyperparameters)

if SAVE_OUTPUT_CSV_FILES:
    hyperparameter_df.to_csv(
        "hyperparameter_experiment_results.csv",
        index=False
    )
    hyperparameter_repeat_df.to_csv(
        "hyperparameter_experiment_repeat_results.csv",
        index=False
    )



&lt;h2 id="Final-optimized-multimodal-model"&gt;Final optimized multimodal model<a class="anchor-link" href="#Final-optimized-multimodal-model">¶</a>&lt;/h2&gt;&lt;p&gt;After feature and hyperparameter selection are complete, the chosen configuration is refitted on the entire outer training set. The number of epochs is taken from the median best epoch observed during repeated validation.&lt;/p&gt;
&lt;p&gt;Only then is the reserved test set evaluated once to report the final generalisation metrics.&lt;/p&gt;



In [ ]:
# Final optimized multimodal model using validation-selected FRED features
# and validation-selected hyperparameters.

# Preserve the candidate expanded-FRED multimodal outputs for comparison.
expanded_fusion_metrics = fusion_metrics.copy()
pred_expanded_fusion = pred_fusion.copy()
model_expanded_fusion = model_fusion
expanded_tabular_preprocessor = main_tabular_preprocessor
expanded_tabular_cols = tabular_cols.copy()

# Use the setting selected by repeated validation, never by test RMSE.
best_params = {
    "lstm_units": int(best_hyperparameters["LSTM_Units"]),
    "tab_dense_1": int(best_hyperparameters["Tab_Dense_1"]),
    "tab_dense_2": int(best_hyperparameters["Tab_Dense_2"]),
    "fusion_dense": int(best_hyperparameters["Fusion_Dense"]),
    "dropout": float(best_hyperparameters["Dropout"]),
    "learning_rate": float(best_hyperparameters["Learning_Rate"]),
    "batch_size": int(best_hyperparameters["Batch_Size"]),
}

final_training_epochs = max(
    1,
    int(round(best_hyperparameters["Median_Best_Epoch"])),
)

print("Training final optimized model with parameters:")
print(best_params)
print("Final refit epochs selected from repeated validation:", final_training_epochs)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model_final_fusion = build_multimodal_model(
    input_tab_dim=X_train_tab_final.shape[1],
    params=best_params,
)

# No validation_split and no test monitoring are used during this final refit.
history_final_fusion = model_final_fusion.fit(
    [X_train_tab_final, X_train_seq],
    y_train_seq,
    epochs=final_training_epochs,
    batch_size=best_params["batch_size"],
    verbose=1,
    shuffle=False,
)

# First and only final-model evaluation on the reserved test set.
pred_final_fusion_scaled = model_final_fusion.predict(
    [X_test_tab_final, X_test_seq],
    verbose=0,
).flatten()

pred_final_fusion = scaler_y.inverse_transform(
    pred_final_fusion_scaled.reshape(-1, 1)
).flatten()
final_fusion_metrics = regression_metrics(y_test_real, pred_final_fusion)

print("Final optimized multimodal TEST metrics:")
print(final_fusion_metrics)

# Make the optimized model the default model used by plots and prediction.
model_fusion = model_final_fusion
X_train_tab_processed_fusion = X_train_tab_final
X_test_tab_processed_fusion = X_test_tab_final
tabular_cols = optimized_tabular_cols
exogenous_cols = selected_fred_cols
pred_fusion = pred_final_fusion
fusion_metrics = final_fusion_metrics
rmse_mm_tab_time = fusion_metrics["RMSE"]
mae_mm_tab_time = fusion_metrics["MAE"]

results_df = pd.DataFrame([
    {"Model": "Linear Regression", **lr_metrics},
    {"Model": "MLP", **mlp_metrics},
    {"Model": "XGBoost (History)", **xgb_metrics},
    {"Model": "XGBoost–LSTM", **xgb_lstm_metrics},
    {"Model": "LSTM", **lstm_metrics},
    {"Model": "Candidate MM(Expanded FRED)", **expanded_fusion_metrics},
    {"Model": "Final MM(Ablation-selected FRED)", **final_fusion_metrics},
]).sort_values("RMSE").reset_index(drop=True)

print("Updated model comparison including final optimized model:")
print(results_df)

if SAVE_OUTPUT_CSV_FILES:
    results_df.to_csv("final_model_comparison_results.csv", index=False)



&lt;h2 id="Plots-and-interpretation-helpers"&gt;Plots and interpretation helpers<a class="anchor-link" href="#Plots-and-interpretation-helpers">¶</a>&lt;/h2&gt;&lt;p&gt;The following cells produce figures for the report/dissertation.&lt;/p&gt;



In [ ]:
# Shared setup for all report figures.
model_predictions = {
    "Linear Regression": pred_lr,
    "MLP": pred_mlp,
    "XGBoost (History)": pred_xgb,
    "XGBoost–LSTM": pred_xgb_lstm,
    "LSTM": pred_lstm,
    "Candidate MM(Expanded FRED)": pred_expanded_fusion,
    "Final MM(Ablation-selected FRED)": pred_fusion,
}

y_true_common = y_test_real
model_names_ordered = list(model_predictions.keys())
model_colors = dict(zip(
    model_names_ordered,
    plt.cm.tab10(np.linspace(0, 1, len(model_names_ordered))),
))

# The outer test split is random for model evaluation. For visualisation only,
# reorder held-out rows chronologically by target year/month and then hotel_id.
# This is a chronological ordering of held-out samples, not a continuous
# single-hotel time series.
test_plot_metadata = aligned_df.iloc[test_idx][
    ["target_year", "target_month", "hotel_id"]
].copy()
test_plot_metadata["_test_position"] = np.arange(len(test_idx))
test_plot_metadata = test_plot_metadata.sort_values(
    ["target_year", "target_month", "hotel_id"]
).reset_index(drop=True)

n_show = min(100, len(test_plot_metadata))
plot_positions = test_plot_metadata["_test_position"].to_numpy()[:n_show]

actual_plot_values = y_true_common[plot_positions]
all_price_plot_values = np.concatenate(
    [actual_plot_values]
    + [model_predictions[name][plot_positions] for name in model_names_ordered]
)
price_min = float(np.nanmin(all_price_plot_values))
price_max = float(np.nanmax(all_price_plot_values))
price_margin = max((price_max - price_min) * 0.05, 1.0)
common_price_ylim = (price_min - price_margin, price_max + price_margin)

abs_errors = {
    name: np.abs(actual_plot_values - model_predictions[name][plot_positions])
    for name in model_names_ordered
}
common_error_max = max(float(values.max()) for values in abs_errors.values())
common_error_ylim = (0.0, max(common_error_max * 1.05, 1.0))

print("Models available for plotting:", model_names_ordered)
print("Plot rows are chronologically ordered held-out samples, not one continuous hotel series.")
print("Shared hotel-price y-axis:", common_price_ylim)


In [ ]:
# Final metric comparison figures: one independent figure per metric (vertical bars)
for metric in ["RMSE", "MAE", "MAPE", "R2"]:
    ascending = metric != "R2"  # lower is better for RMSE/MAE/MAPE; higher is better for R2
    ordered = results_df.sort_values(metric, ascending=ascending).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    bars = ax.bar(ordered["Model"], ordered[metric])
    ax.set_ylabel(metric if metric != "MAPE" else "MAPE (%)")
    ax.set_xlabel("Models")
    ax.set_title(f"Final Model Comparison: {metric}")
    ax.grid(axis="y", alpha=0.25)
    ax.set_xticklabels(ordered["Model"], rotation=20, ha="right")

    max_value = float(ordered[metric].max())
    offset = max(abs(max_value) * 0.015, 0.01)
    for bar, value in zip(bars, ordered[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# Dataset overview figures

# 1. Train / test split composition
split_summary_df = pd.DataFrame([
    {"Subset": "Train", "Samples": len(train_idx)},
    {"Subset": "Test", "Samples": len(test_idx)},
])
split_summary_df["Percentage"] = split_summary_df["Samples"] / split_summary_df["Samples"].sum() * 100

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(split_summary_df["Subset"], split_summary_df["Samples"])
ax.invert_yaxis()
ax.set_xlabel("Number of samples")
ax.set_title("Composition of the Supervised Dataset")
ax.grid(axis="x", alpha=0.25)
for bar, samples, pct in zip(bars, split_summary_df["Samples"], split_summary_df["Percentage"]):
    ax.text(
        bar.get_width() + split_summary_df["Samples"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{samples:,} ({pct:.0f}%)",
        va="center",
    )
plt.tight_layout()
plt.show()
plt.close(fig)

# 2. Target-price distribution in the central 99% range, train vs test
price_limit = float(np.percentile(y_tab, 99))
fig, ax = plt.subplots(figsize=(9, 5))
for label, values in [("Train", y_train_tab), ("Test", y_test_tab)]:
    shown = values[values <= price_limit]
    ax.hist(shown, bins=50, alpha=0.5, label=label)
ax.set_xlabel("Target hotel price")
ax.set_ylabel("Frequency")
ax.set_title("Target-Price Distribution (Central 99%)")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

# 3. Top-10 city composition, where available
if "city" in aligned_df.columns:
    city_counts = aligned_df["city"].astype(str).value_counts().head(10)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(city_counts.index, city_counts.values)
    ax.invert_yaxis()
    ax.set_xlabel("Supervised samples")
    ax.set_title("Top 10 Cities in the Supervised Dataset")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# 4. Final model input dimensions
input_dimensions = pd.Series({
    "Historical price steps": sequence_length,
    "Selected FRED columns": len(exogenous_cols),
    "Raw tabular columns": len(base_tabular_cols),
    "Processed tabular features": X_train_tab_final.shape[1],
})
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(input_dimensions.index, input_dimensions.values)
ax.invert_yaxis()
ax.set_xlabel("Dimension")
ax.set_title("Final Model Input Dimensions")
ax.grid(axis="x", alpha=0.25)
for bar, value in zip(bars, input_dimensions.values):
    ax.text(value + max(input_dimensions.values) * 0.01, bar.get_y() + bar.get_height() / 2, str(value), va="center", fontsize=9)
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# Actual vs Predicted: all models combined on chronologically ordered held-out rows
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(actual_plot_values, label="Actual", linewidth=2)
for name in model_names_ordered:
    ax.plot(
        model_predictions[name][plot_positions],
        label=name,
        color=model_colors[name],
        alpha=0.85,
    )
ax.set_ylim(*common_price_ylim)
ax.set_title(
    f"Actual versus Predicted Hotel Prices: All Models "
    f"({n_show} Chronologically Ordered Held-Out Samples)"
)
ax.set_xlabel("Chronologically Ordered Held-Out Sample Index")
ax.set_ylabel("Hotel Price")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# Actual vs Predicted: one independent figure per model
# Every panel uses exactly the same y-axis range for fair visual comparison.
for name in model_names_ordered:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(actual_plot_values, label="Actual", linewidth=2)
    ax.plot(
        model_predictions[name][plot_positions],
        label=f"Predicted: {name}",
        color=model_colors[name],
    )
    ax.set_ylim(*common_price_ylim)
    ax.set_title(f"Actual versus Predicted Hotel Prices: {name}")
    ax.set_xlabel("Chronologically Ordered Held-Out Sample Index")
    ax.set_ylabel("Hotel Price")
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# Absolute prediction error: all models combined
fig, ax = plt.subplots(figsize=(12, 6))
for name in model_names_ordered:
    ax.plot(abs_errors[name], label=name, color=model_colors[name])
ax.set_ylim(*common_error_ylim)
ax.set_title(
    f"Absolute Prediction Errors: All Models "
    f"({n_show} Chronologically Ordered Held-Out Samples)"
)
ax.set_xlabel("Chronologically Ordered Held-Out Sample Index")
ax.set_ylabel("Absolute Error")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# Absolute prediction error: one independent figure per model
# Every panel uses exactly the same error-axis range.
for name in model_names_ordered:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(abs_errors[name], color=model_colors[name], label=f"{name} absolute error")
    ax.set_ylim(*common_error_ylim)
    ax.set_title(f"Absolute Prediction Error: {name}")
    ax.set_xlabel("Chronologically Ordered Held-Out Sample Index")
    ax.set_ylabel("Absolute Error")
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# Residual distribution: all models combined
residuals = {name: y_true_common - model_predictions[name] for name in model_names_ordered}

residual_limit = 250
fig, ax = plt.subplots(figsize=(10, 6))
for name in model_names_ordered:
    err = residuals[name]
    shown = err[np.abs(err) < residual_limit]
    ax.hist(shown, bins=50, alpha=0.45, label=name, color=model_colors[name])
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Zero residual")
ax.set_title("Residual Distributions: All Models")
ax.set_xlabel("Residual: Actual Price - Predicted Price")
ax.set_ylabel("Frequency")
ax.set_xlim(-residual_limit, residual_limit)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# Residual distribution: one independent figure per model
for name in model_names_ordered:
    err = residuals[name]
    shown = err[np.abs(err) < residual_limit]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.hist(shown, bins=50, alpha=0.75, color=model_colors[name], label=name)
    ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Zero residual")
    mean_resid = float(err.mean())
    ax.axvline(mean_resid, color="blue", linestyle=":", linewidth=1, label=f"Mean residual = {mean_resid:.2f}")
    ax.set_title(f"Residual Distribution: {name}")
    ax.set_xlabel("Residual: Actual Price - Predicted Price")
    ax.set_ylabel("Frequency")
    ax.set_xlim(-residual_limit, residual_limit)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# Predict next-month hotel price using the final optimized multimodal model

import warnings

# Columns that describe the future search/booking SCENARIO for the target month
# (as opposed to a fairly static hotel characteristic like distance/stars/rating).
#
# During training (see aligned_df construction), these columns are taken from the
# TARGET row -- i.e. the actual month being predicted -- NOT from the forecast-origin
# row. That means at prediction time they should describe the scenario you want to
# forecast for (e.g. "a 2-night weekend stay with no special offer"), not whatever
# happened to be true in the hotel's most recently observed month. If they are not
# supplied explicitly, falling back to the latest known month's value is only a rough
# approximation and is flagged as such below.
SCENARIO_TABULAR_COLS = [
    c for c in ["weekend", "holiday", "nnights", "scarce_room", "offer", "offer_cat"]
    if c in base_tabular_cols
]


def predict_next_month_price(hotel_id, future_feature_overrides=None, strict_scenario_check=False):
    """
    Predict next-month hotel price for one hotel using the final ablation-selected multimodal model.

    Parameters
    ----------
    hotel_id : selected hotel_id from df_full
    future_feature_overrides : optional dict
        Used to set the actual future search/booking scenario for the target month,
        e.g. weekend, holiday, nnights, scarce_room, offer, offer_cat.
        IMPORTANT: during training these columns come from the TARGET month's row (the
        row that is actually being predicted), not from the forecast-origin month. If you
        don't supply them here, this function falls back to the latest known month's
        values as a rough approximation and raises a warning, because that fallback does
        not match how the model was trained.
    strict_scenario_check : bool, default False
        If True, raise a ValueError instead of a warning when scenario columns
        (see SCENARIO_TABULAR_COLS) are missing from future_feature_overrides.

    Returns
    -------
    dict containing hotel_id, forecast origin, target month, which scenario columns (if
    any) were approximated from the latest known month, and the predicted price.
    """

    if future_feature_overrides is None:
        future_feature_overrides = {}

    missing_scenario_cols = [
        c for c in SCENARIO_TABULAR_COLS if c not in future_feature_overrides
    ]

    if missing_scenario_cols:
        message = (
            "The following search/booking scenario features were not provided in "
            f"future_feature_overrides: {missing_scenario_cols}. "
            "During training these columns are taken from the TARGET month's row, not "
            "the forecast-origin month, so falling back to the latest known month's "
            "values is only a rough approximation of the future scenario and may not "
            "match the actual target-month search context. Pass explicit values in "
            "future_feature_overrides for a more reliable forecast."
        )
        if strict_scenario_check:
            raise ValueError(message)
        warnings.warn(message, stacklevel=2)

    # 1. Get this hotel's historical records
    group = df_full[df_full["hotel_id"] == hotel_id].sort_values(["year", "month"]).reset_index(drop=True)

    if len(group) < sequence_length:
        raise ValueError(f"Hotel {hotel_id} does not have enough history for sequence_length={sequence_length}.")

    # 2. Use the latest available month as forecast origin
    latest_row = group.iloc[-1]

    origin_year = int(latest_row["year"])
    origin_month = int(latest_row["month"])

    # 3. Calculate next month
    origin_period = pd.Period(year=origin_year, month=origin_month, freq="M")
    target_period = origin_period + prediction_horizon

    target_year = target_period.year
    target_month = target_period.month

    # 4. Use the latest sequence_length prices as LSTM input
    hist_prices = group["price"].tail(sequence_length).values

    X_seq_future = hist_prices.reshape(1, sequence_length, 1)
    X_seq_future_scaled = scaler_x.transform(
        X_seq_future.reshape(-1, 1)
    ).reshape(1, sequence_length, 1)

    # 5. Build future tabular row
    future_row = {}

    for col in base_tabular_cols:
        if col == "year":
            future_row[col] = target_year
        elif col == "month":
            future_row[col] = target_month
        else:
            # Static hotel characteristics (distance, stars, rating, city, ...) fall back
            # to the latest known value, which is a reasonable approximation since these
            # rarely change month to month. Scenario columns (SCENARIO_TABULAR_COLS) also
            # fall back here by default, but that is only an approximation -- see the
            # warning/check above -- and should normally be supplied via
            # future_feature_overrides instead.
            future_row[col] = latest_row[col]

    # FRED variables use latest available month to avoid future leakage
    for col in exogenous_cols:
        future_row[col] = latest_row[col]

    # Manually override future features if provided
    for key, value in future_feature_overrides.items():
        future_row[key] = value

    X_future_raw = pd.DataFrame([future_row])

    # 6. Apply the final train-fitted preprocessing pipeline directly.
    # No target/test rows are concatenated and no category vocabulary is re-fitted.
    X_future_raw = X_future_raw.reindex(columns=optimized_tabular_cols)
    X_future_raw = X_future_raw.replace([np.inf, -np.inf], np.nan)
    X_future_tab_processed = final_tabular_preprocessor.transform(X_future_raw)
    X_future_tab_processed = np.asarray(X_future_tab_processed, dtype=np.float32)

    # 8. Predict
    pred_scaled = model_fusion.predict(
        [X_future_tab_processed, X_seq_future_scaled],
        verbose=0
    ).flatten()

    pred_price = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()[0]

    return {
        "hotel_id": hotel_id,
        "origin_year": origin_year,
        "origin_month": origin_month,
        "target_year": target_year,
        "target_month": target_month,
        "model_used": "Final MM(Ablation-selected FRED)",
        "selected_ablation_feature_set": selected_ablation_feature_set,
        "num_selected_fred_cols": len(selected_fred_cols),
        "approximated_scenario_cols": missing_scenario_cols,
        "predicted_next_month_price": pred_price
    }


In [ ]:
# Example: explicitly specify the future search/booking scenario for the target month
# (weekend/holiday/nnights/scarce_room/offer/offer_cat), instead of letting the function
# silently fall back to the hotel's most recently observed scenario.
prediction = predict_next_month_price(
    hotel_id=12345,
    future_feature_overrides={
        "weekend": 1,
        "holiday": 0,
        "nnights": 2,
        "scarce_room": 0,
        "offer": 0,
        # Replace "0% no offer" below with one of the actual offer_cat
        # categories present in your own aligned_df["offer_cat"].unique().
        "offer_cat": "0% no offer"
    }
)

# "approximated_scenario_cols" lists any scenario columns that were NOT provided above
# and therefore fell back to the latest known month's value as a rough approximation.
# It should be empty here since all scenario columns were supplied explicitly.
prediction
